# Model to transform infrastructure

- enables to transform the methane into a hydrogen network
- allows for new investments on defined edges
- bidirectional net flows on edges possible

### Import packages

In [38]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import networkx as nx

### Import data

In [39]:
# Specify the path to your Excel file
input_file_path = '../01_data/01_input_data/02_processed/'
excel_file_path = 'Data_update.xlsx'  

# Read the Excel file into a DataFrame
df_nodes = pd.read_excel(input_file_path + excel_file_path, sheet_name='Nodes')
df_commodities = pd.read_excel(input_file_path + excel_file_path, sheet_name='Commodities')
df_edges = pd.read_excel(input_file_path + excel_file_path, sheet_name='Edges')
df_parameter = pd.read_excel(input_file_path + excel_file_path, sheet_name='Parameters')
df_supply_values = pd.read_excel(input_file_path + excel_file_path, sheet_name='Supply')

### Create input data structure

In [40]:
#commodities
commodities = df_commodities['Commodities'].tolist()

#nodes
nodes = df_nodes['Nodes'].tolist()

#edges
edges_base = list(zip(df_edges['Source'], df_edges['Destination']))

#initial capacities
capacity_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['initial_capacities'] 
                 for _, row in df_parameter.iterrows()}

#maximal capacities
capacity_limit_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['max_capacities'] 
                 for _, row in df_parameter.iterrows()}

#variable transportation cost
cost_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['costs_edge'] 
                 for _, row in df_parameter.iterrows()}

#investment cost new capacity
investment_cost_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['new_build_cost'] 
                 for _, row in df_parameter.iterrows()}

#investment cost convert capacity
conversion_cost_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['conversion_cost'] 
                 for _, row in df_parameter.iterrows()}

#converseion capacity factor
conversion_capacity_factor = {tuple(row[['Commodity', 'Source', 'Destination']]): row['conversion_capacity_factor'] 
                 for _, row in df_parameter.iterrows()}

#demand and supply
inflow = {tuple(row[['Commodity', 'Node']]): row['Supply'] 
                 for _, row in df_supply_values.iterrows()}

### adjust input data

In [41]:
# Reverse edges
edges_reverse = [(destination, source) for source, destination in edges_base]

# Combined edges
edges = edges_base + edges_reverse

# Reverse cap
capacity_reverse = {(k, j, i): cap for (k, i, j), cap in capacity_base.items()}

capacity = {**capacity_base, **capacity_reverse}

# Reverse cap
capacity_limit_reverse = {(k, j, i): cap for (k, i, j), cap in capacity_limit_base.items()}

capacity_limit = {**capacity_limit_base, **capacity_limit_reverse}

# Reverse cost
cost_reverse = {(k, j, i): cost for (k, i, j), cost in cost_base.items()}

cost = {**cost_base, **cost_reverse}

# Reverse cost
investment_cost_reverse = {(k, j, i): cost for (k, i, j), cost in investment_cost_base.items()}

investment_cost = {**investment_cost_base, **investment_cost_reverse}

# Reverse cost
conversion_cost_reverse = {(k, j, i): cost for (k, i, j), cost in conversion_cost_base.items()}

conversion_cost = {**conversion_cost_base, **conversion_cost_reverse}


## Model

### Create model

In [42]:
# Create optimization model
m = gp.Model("Grid_Transformation")

### Define variables

In [43]:
# Create variables
flow = m.addVars(commodities, edges, vtype=GRB.CONTINUOUS, name="flow")
# variable for capacity investment
investment = m.addVars(commodities, edges, vtype=GRB.CONTINUOUS, name="investment")
# Variable for converted capacity
converted_capacity = m.addVars(commodities, edges, vtype=GRB.CONTINUOUS, name="converted_capacity")
# Binary variable to indicate if conversion is active
conversion_indicator = m.addVars(commodities, edges, vtype=GRB.BINARY, name="conversion_indicator")

### Define objective

In [44]:
# Set objective function
m.setObjective(
    gp.quicksum(flow[h, i, j] * cost[h, i, j] for h in commodities for i, j in edges) +
    gp.quicksum(investment[h, i, j] * investment_cost[h, i, j] for h in commodities for i, j in edges_base) +
    gp.quicksum(converted_capacity[h, i, j] * conversion_cost[h, j, i] for h in commodities for i, j in edges_base),
    GRB.MINIMIZE
)

### Define constraints

In [45]:
# Edge-capacity constraints with investment
m.addConstrs((flow[h, i, j] <= converted_capacity[h, i, j] + investment[h, i, j] for h in commodities for i, j in edges), "cap")

# Edge-capacity total limit
m.addConstrs((investment[h, i, j] + converted_capacity[h, i, j] 
              <= capacity_limit[h, i, j] for h in commodities for i, j in edges_base), "cap")

# Investment constraints
m.addConstrs((investment[h, i, j] == investment[h, j, i] for h in commodities for i, j in edges), "conversion")

#conversion binary constraint
m.addConstrs((gp.quicksum(conversion_indicator[h, i, j] for h in commodities) == 1 for i, j in edges), "conversion_indicator_sum_one")

#link the converted capacity to the initial capacity
m.addConstrs((capacity[commodities[0], i, j] * conversion_indicator[h, i, j] 
              == converted_capacity[h, i, j] for h in commodities[1:] for i, j in edges), "capacity_relationship")
m.addConstrs((capacity[commodities[0], i, j] * conversion_indicator[commodities[0], i, j]
        == converted_capacity[commodities[0], i, j]
        for i, j in edges), "capacity_relationship_first_commodity")

# Conversion constraints
m.addConstrs((converted_capacity[h, i, j] == converted_capacity[h, j, i] for h in commodities for i, j in edges), "conversion")

# Flow-conservation constraints
m.addConstrs(
    (
        flow.sum(h, "*", j) + inflow[h, j] == flow.sum(h, j, "*")
        for h in commodities
        for j in nodes
    ),
    "node",
)

print('Constraints createt')

Constraints createt


### Optimize the model

In [46]:
# Compute optimal solution
m.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 306 rows, 224 columns and 728 nonzeros
Model fingerprint: 0x66b7893c
Variable types: 168 continuous, 56 integer (56 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [5e-01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]
Found heuristic solution: objective 10401.000000
Presolve removed 271 rows and 184 columns
Presolve time: 0.00s
Presolved: 35 rows, 40 columns, 116 nonzeros
Found heuristic solution: objective 6464.0000000
Variable types: 34 continuous, 6 integer (6 binary)

Root relaxation: objective 4.589000e+03, 18 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth In

## Outputs

In [47]:
if m.Status == GRB.OPTIMAL:
    solution = m.getAttr("X", flow)
    investment_solution = m.getAttr("X", investment)
    converted_capacity_solution = m.getAttr("X", converted_capacity)
    conversion_indicator_solution = m.getAttr("X", conversion_indicator)

    print("****************************")
    print(f"Total cost: {m.objVal}")
    print("****************************")
    
    for h in commodities:
        print(f"\nOptimal flows for {h}:")
        for i, j in edges:
            if solution[h, i, j] > 0:
                print(f"{i} -> {j}: {solution[h, i, j]:g}")

        print(f"\nOptimal investments for {h}:")
        for i, j in edges_base:
            if investment_solution[h, i, j] > 0:
                print(f"Investment for {h} in edge ({i} -> {j}): {investment_solution[h, i, j]:g}")

        print(f"\nOptimal converted capacities for {h}:")
        for i, j in edges_base:
            if converted_capacity_solution[h, i, j] > 0:
                print(f"Converted capacity for {h} from {i} to {j}: {converted_capacity_solution[h, i, j]:g}")

    print("\nOptimal conversion indicators:")
    for h in commodities:
        for i, j in edges_base:
            #
                print(f"Conversion of {h} from {i} to {j} is: {conversion_indicator_solution[h, i, j]:g}")
else:
    print("No optimal solution found.")

****************************
Total cost: 5459.0
****************************

Optimal flows for Methane:
S1 -> D1: 20
S2 -> D3: 80
S3 -> D1: 15
D1 -> D2: 20
D3 -> D4: 55
D4 -> D5: 44
D4 -> D6: 10
D6 -> D7: 10
D5 -> D8: 10
D5 -> D9: 10
D7 -> D10: 5

Optimal investments for Methane:
Investment for Methane in edge (S2 -> D3): 80
Investment for Methane in edge (D3 -> D4): 55
Investment for Methane in edge (D4 -> D5): 44

Optimal converted capacities for Methane:
Converted capacity for Methane from S1 to D1: 500
Converted capacity for Methane from S1 to D2: 500
Converted capacity for Methane from S2 to D2: 500
Converted capacity for Methane from S3 to D1: 500
Converted capacity for Methane from D1 to D2: 500
Converted capacity for Methane from D4 to D6: 500
Converted capacity for Methane from D6 to D7: 500
Converted capacity for Methane from D5 to D8: 500
Converted capacity for Methane from D5 to D9: 500
Converted capacity for Methane from D9 to D10: 500
Converted capacity for Methane from 